# Notebook Requirements
Run this code to execute the notebook if you didn't already cloned the repo.

In [ ]:
!git clone https://github.com/GiuseppeDaddario/Computer-Vision.git --recurse-submodules
%cd Computer-Vision

# Imports

In [ ]:
%pip install ultralytics --quiet
%pip install -U gdown

In [ ]:
import os
os.environ['WANDB_MODE'] = 'disabled'
from pathlib import Path
from tqdm import tqdm
import shutil
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
import random
import numpy as np
#from src import YOLOv5_training, YOLOv5_inference #Importing from the yolov5 repo

# Globals

In [ ]:
DATASET_PATH = "dataset/CCPD2019"
DATASET_PATH_YOLO = "dataset/CCPD2019_YOLO"

# YOLOv5 paths
TRAINING_PATH_YOLO = "dataset/ccpd_2019.yaml"
test_dir = "ccpd_challenge"
TEST_PATH_YOLOV5 = f"dataset/CCPD_YOLO/{test_dir}/images/test" 
PROJECT_PATH = '/leonardo/home/userexternal/gdaddari/Computer-Vision/src/YOLO/runs'
YOLO_MODEL_PATH = 'src/YOLO/runs/train/weights/best.pt'


#TODO Lore: write your paths
TRAINING_PATH_PDLPR = ""
TEST_PATH_PDLPR = ""

IMG_WIDTH = 1160
IMG_HEIGHT = 720
CLASS_ID = 0 

# Utils

## Baseline

In [ ]:
##################### UTILS FOR THE BASELINE #####################

##################################################################

## YOLOv5

In [ ]:
######################## UTILS FOR YOLOV5 ########################

def convert_bbox(x1, y1, x2, y2):
    """
    Converts bbox coordinates in pixels (normalized), following the YOLO format.
    """
    bbox_width = abs(x2 - x1)
    bbox_height = abs(y2 - y1)
    x_center = x1 + bbox_width / 2.0
    y_center = y1 + bbox_height / 2.0

    # Normalizing
    x_center /= IMG_WIDTH
    y_center /= IMG_HEIGHT
    bbox_width /= IMG_WIDTH
    bbox_height /= IMG_HEIGHT

    return x_center, y_center, bbox_width, bbox_height

def parse_filename(fname):
    """
    Extracts bbox coordinates from the image file name and converts them in YOLO format
    """
    fname = Path(fname)
    parts = fname.stem.split('-')
    if len(parts) != 7:
        return None

    try:
        bbox_str = parts[2]
        x1y1_str, x2y2_str = bbox_str.split('_')
        x1, y1 = map(int, x1y1_str.split('&'))
        x2, y2 = map(int, x2y2_str.split('&'))

        return convert_bbox(x1, y1, x2, y2)
    except Exception as e:
        print(f"[WARN] Could not parse bbox from file '{fname}': {e}")
        return None

def process_images(images, images_src, dest_root, split):
    """
    Copies images in a new folder building the structure (splits and subfolders) required by YOLOv5.
    """
    images_dest = Path(dest_root) / "images" / split
    labels_dest = Path(dest_root) / "labels" / split
    os.makedirs(images_dest, exist_ok=True)
    os.makedirs(labels_dest, exist_ok=True)

    for img_path in tqdm(images, desc=f"Processing {split} set"):
        bbox = parse_filename(img_path.name)
        if bbox is None:
            continue

        shutil.copy(img_path, images_dest / img_path.name)
        label_path = labels_dest / (img_path.stem + ".txt")
        with open(label_path, 'w') as f:
            f.write(f"{CLASS_ID} {' '.join(f'{x:.6f}' for x in bbox)}\n")

    print(f"{split} set saved to {images_dest} and {labels_dest}")

def prepare_ccpd_base(dest_root="CCPD_YOLO", split_ratio=0.8, seed=42):
    """
    Builds the training subdataset 'ccpd_base'.
    """
    src = "ccpd_base"
    images_src = Path(f"dataset/CCPD2019/{src}")
    image_files = list(images_src.glob("*.jpg"))
    random.seed(seed)
    random.shuffle(image_files)

    split_index = int(len(image_files) * split_ratio)
    train_files = image_files[:split_index]
    val_files = image_files[split_index:]

    process_images(train_files, images_src, f"dataset/{dest_root}/{src}", "train")
    process_images(val_files, images_src, f"dataset/{dest_root}/{src}", "val")

def prepare_other_subset(subset, dest_root="CCPD_YOLO"):
    """
    Builds the other subdatasets for the testing phase (individually).
    """
    images_src = Path(f"dataset/CCPD2019/{subset}")
    image_files = list(images_src.glob("*.jpg"))
    process_images(image_files, images_src, f"dataset/{dest_root}/{subset}", "test")

##################################################################

## PDLPR

In [ ]:
######################## UTILS FOR PDLPR #########################
#TODO Lore: move this functions directly in the notebook
from src import PDLPR_training, PDLPR_inference
##################################################################

# Data

In [ ]:
SO="MacOs"
# Installing pixz for faster unxipping
if SO=="Linux":
    !apt-get update
    !apt-get install pixz
elif SO=="MacOs":
    !brew install pixz

zsh:1: command not found: apt-get


In [ ]:
# Downloading the .tar dataset and extracting it
%cd dataset
!gdown --id 1HDyFIuH65kVLtsXqxLA8gs0gJr7CRynh
!tar -I 'pixz -d' -xf CCPD2019.tar.xz

In [ ]:
#TODO Lore: move here the class for the dataset so that can be used in the every part of the code

## Baseline

## YOLOv5

In [ ]:
base_dest = "CCPD_YOLO"

other_subsets = [
    "ccpd_blur", "ccpd_challenge", "ccpd_db",
    "ccpd_fn", "ccpd_np", "ccpd_rotate", "ccpd_tilt", "ccpd_weather"
]

# Build the training subset (both train-val splits)
prepare_ccpd_base(dest_root=base_dest)

# Other subsets (only for testing)
for subset in other_subsets:
    prepare_other_subset(subset, dest_root=base_dest)

## PDLRP

In [ ]:
#TODO Lore: move here the actual preprocessing needed (the function calls)

# Network

## Baseline

## YOLOv5

## PDLRP

In [ ]:
#TODO Lore: move here the model architecture

# Train

## Baseline

## YOLOv5

In [ ]:
# Train YOLOv5s
YOLOv5_training(
    weights="yolov5s.pt",
    data=DATASET_PATH_YOLO,
    epochs=300,
    batch_size=50,
    imgsz=640,
    optimizer="Adam",
    lr0=1e-3,
    lrf=1e-5,
    cos_lr=True,
    project="runs/train",
    name="lp_detection",
    cache="ram"
)

## PDLRP

In [ ]:
# ------ training ------ #
#TODO Lore: check this works
print("PDLPR Training ...")
PDLPR_training(TRAINING_PATH_PDLPR, batch_size=32, num_epochs=3)

# Evaluation

## Baseline

## YOLOv5

In [ ]:
YOLOv5_inference(
    weights=YOLO_MODEL_PATH,
    source=TEST_PATH_YOLOV5,
    imgsz=640,
    device="cuda:0",
    project=PROJECT_PATH,
    name="test",
    exist_ok=True
)

## PDLRP

In [ ]:
# ------ inference ------ #
#TODO Lore: check this works
print("PDLPR Inference ...")
PDLPR_inference(TEST_PATH_PDLPR, batch_size=64)